# Project 03: QSVM vs Classical SVM

This notebook compares a classical SVM with a quantum-kernel model on low-dimensional features.

In [ ]:
# Imports
import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel

## 1) Load and Filter Dataset

In [ ]:
digits = load_digits()
X = digits.data
y = digits.target

# Keep only 2 classes first for stable baseline (e.g., 0 vs 1)
mask = (y == 0) | (y == 1)
X = X[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

## 2) Dimensionality Reduction (Critical for Quantum)

In [ ]:
n_qubits = 4

scaler = StandardScaler()
pca = PCA(n_components=n_qubits, random_state=42)

X_train_red = pca.fit_transform(scaler.fit_transform(X_train))
X_test_red = pca.transform(scaler.transform(X_test))

# Optional feature scaling to [0, pi] for angle encoding
X_min = X_train_red.min(axis=0)
X_max = X_train_red.max(axis=0)
X_train_q = np.pi * (X_train_red - X_min) / (X_max - X_min + 1e-12)
X_test_q = np.pi * (X_test_red - X_min) / (X_max - X_min + 1e-12)

## 3) Classical SVM Baseline

In [ ]:
clf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)

t0 = time.time()
clf.fit(X_train_red, y_train)
classical_time = time.time() - t0
classical_pred = clf.predict(X_test_red)
classical_acc = accuracy_score(y_test, classical_pred)

print(f"Classical SVM accuracy: {classical_acc:.4f}")
print(f"Classical train time: {classical_time:.3f}s")
print(classification_report(y_test, classical_pred))

## 4) Quantum Kernel Model

In [ ]:
feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2, entanglement="linear")
sampler = StatevectorSampler()
qkernel = FidelityQuantumKernel(feature_map=feature_map, sampler=sampler)

# Use precomputed kernels with scikit-learn SVC
t0 = time.time()
K_train = qkernel.evaluate(x_vec=X_train_q)
K_test = qkernel.evaluate(x_vec=X_test_q, y_vec=X_train_q)

qsvc = SVC(kernel="precomputed")
qsvc.fit(K_train, y_train)
quantum_time = time.time() - t0

quantum_pred = qsvc.predict(K_test)
quantum_acc = accuracy_score(y_test, quantum_pred)

print(f"Quantum-kernel accuracy: {quantum_acc:.4f}")
print(f"Quantum kernel pipeline time: {quantum_time:.3f}s")
print(classification_report(y_test, quantum_pred))

## 5) Comparison

In [ ]:
labels = ["Classical SVM", "Quantum Kernel SVM"]
acc = [classical_acc, quantum_acc]
times = [classical_time, quantum_time]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(labels, acc)
ax[0].set_title("Accuracy")
ax[0].set_ylim(0, 1)

ax[1].bar(labels, times)
ax[1].set_title("Runtime (s)")

for axis in ax:
    axis.tick_params(axis="x", rotation=20)
    axis.grid(alpha=0.2)

plt.tight_layout()
plt.show()